# MovieLens playground

Local look at `ml-latest-small`. Raw files live under `data/raw_private/movielens/` (gitignored).

Use this notebook to poke around: shapes, joins to `tmdbId`, genres, rating quirks.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("..") / "data" / "raw_private" / "movielens" / "ml-latest-small"

movies = pd.read_csv(DATA / "movies.csv")
ratings = pd.read_csv(DATA / "ratings.csv")
tags = pd.read_csv(DATA / "tags.csv")
links = pd.read_csv(DATA / "links.csv")

for name, df in [
    ("movies", movies),
    ("ratings", ratings),
    ("tags", tags),
    ("links", links),
]:
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols → {list(df.columns)}")

In [ ]:
# Nulls + rating basics
display(links.isna().sum())
print("users:", ratings["userId"].nunique())
print("movies with ≥1 rating:", ratings["movieId"].nunique())
print(
    "rating min / max / mean:",
    ratings["rating"].min(),
    ratings["rating"].max(),
    round(ratings["rating"].mean(), 3),
)
ratings["rating"].value_counts().sort_index()

In [ ]:
# Join movies ↔ TMDb ids (this is the bridge we'll use later)
movies_x = movies.merge(links, on="movieId", how="left")
print("movies missing tmdbId:", movies_x["tmdbId"].isna().sum())
movies_x.head(10)

In [ ]:
# Genres are pipe-separated — explode a bit
genre_counts = (
    movies["genres"]
    .str.split("|")
    .explode()
    .value_counts()
)
genre_counts.head(15)

In [ ]:
# Pick a title and see its ratings + tags (playaround)
q = "Toy Story"
hits = movies_x[movies_x["title"].str.contains(q, case=False, na=False)]
display(hits)

mid = hits.iloc[0]["movieId"]
print("ratings for this movie:", (ratings["movieId"] == mid).sum())
display(ratings[ratings["movieId"] == mid]["rating"].describe())
display(tags[tags["movieId"] == mid].head(20))